<a href="https://colab.research.google.com/github/utk-avi/Neural_Style_Transfer_Implementation/blob/main/Neural_Style_Transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NEURAL STYLE TRANSFER**

What does that even mean ?

Neural style transfer is an optimization technique used to take two images—a content image and a style reference image (such as an artwork by a famous painter)—and blend them together so the output image looks like the content image, but “painted” in the style of the style reference image.
This is implemented by optimizing the output image to match the content statistics of the content image and the style statistics of the style reference image. These statistics are extracted from the images using a convolutional network.

For idea and math behind it refer to thr original paper: https://arxiv.org/pdf/1508.06576.pdf

In [9]:
# Import dependencies
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.utils import save_image

In [10]:
model = models.vgg19(pretrained = True).features

In [11]:
print(model)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17): ReLU(inplace=True)
  (18): MaxPoo

In [12]:
class VGG(nn.Module):

    def __init__(self):
        super(VGG, self).__init__()

        # The list stores indices of VGG layers whose features we want to extract (same as the paper)
        self.chosen_features = ["0", "5", "10", "19", "28"]

        # We use 29 layers of VGG19
        self.model = models.vgg19(pretrained=True).features[:29]

    def forward(self, x):

        features = []

        for layer_num, layer in enumerate(self.model):
            x = layer(x)

            if str(layer_num) in self.chosen_features:
                features.append(x)

        return features

In [13]:
def load_image(image_name):

    image = Image.open(image_name)
    image = loader(image).unsqueeze(0)

    return image.to(device)


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
imsize = 256

loader = transforms.Compose(
    [
        transforms.Resize((imsize,imsize)),
        transforms.ToTensor(),
    ]
)

In [16]:
original_img = load_image("/content/JU.jpg")
style_img = load_image("/content/starrynights.jpg")

In [17]:
generated = original_img.clone().requires_grad_(True)

In [18]:
model = VGG().to(device).eval()

In [19]:
total_steps = 6000
learning_rate = 0.001

In [20]:
alpha = 1
beta = 0.01

In [21]:
optimiser = optim.Adam([generated], lr=learning_rate)

In [22]:
for step in range(total_steps):
    generated_features = model(generated)
    original_img_features = model(original_img)
    style_img_features = model(style_img)

    style_loss = 0
    original_loss = 0

    for gen_features, orig_features, style_features in zip(
        generated_features, original_img_features, style_img_features
    ):
        batch_size, channel, height, width = gen_features.shape

        original_loss += torch.mean((gen_features - orig_features) ** 2)

        G = gen_features.view(channel, height * width).mm(
            gen_features.view(channel, height * width).t()
        )
        S = style_features.view(channel, height * width).mm(
            style_features.view(channel, height * width).t()
        )

        style_loss += torch.mean((G - S) ** 2)

    total_loss = alpha * original_loss + beta * style_loss

    optimiser.zero_grad()
    total_loss.backward()
    optimiser.step()

    if step % 200 == 0:
        print("total_loss:", total_loss.item())
        save_image(generated, "generated.jpg")


total_loss: 169001.234375
total_loss: 42017.76953125
total_loss: 16034.1240234375
total_loss: 5647.7392578125
total_loss: 2977.524169921875
total_loss: 2081.53515625
total_loss: 1604.053466796875
total_loss: 1308.6180419921875
total_loss: 1110.7286376953125
total_loss: 972.0584106445312
total_loss: 872.1882934570312
total_loss: 797.009765625
total_loss: 738.3997802734375
total_loss: 690.2687377929688
total_loss: 650.7915649414062
total_loss: 617.372314453125
total_loss: 588.6053466796875
total_loss: 563.2266235351562
total_loss: 540.4345092773438
total_loss: 520.32666015625
total_loss: 502.2285461425781
total_loss: 485.7109375
total_loss: 470.15313720703125
total_loss: 456.2398376464844
total_loss: 443.6607360839844
total_loss: 432.5712890625
total_loss: 422.35736083984375
total_loss: 412.7959899902344
total_loss: 403.0885925292969
total_loss: 395.9149169921875
